In [ ]:
import random
import numpy as np
import pandas as pd
import sys
import numpy as np
import pandas as pd
sys.path.append('../../')
from collections import defaultdict
from utilities import expand_sequences, print_exams
from cvxopt import matrix, solvers
import json


def nextflu_predict(data_test, mutation_effects, serum_potency={}, virus_avidity={}):
        pred_HI = []
        for ind, row in data_test.iterrows():
            muts = get_mutations(row.serumHA, row.virusHA)
            if len(muts) or len(serum_potency) or len(virus_avidity):
                pred = 0
                pred += serum_potency[row.serumName] if row.serumName in serum_potency.keys() else 0
                pred += virus_avidity[row.virusName] if row.virusName in virus_avidity.keys() else 0
                pred += np.sum([mutation_effects[mut] for mut in muts
                                if (mut in mutation_effects and mutation_effects[mut]>0.0)])
            else:
                pred = 0
            
            pred_HI.append(pred)
        
        return np.array(pred_HI)

def relevant_mutations(train_data, isolates):

        # count how often each mutation separates virus and serum
        mutation_counter = defaultdict(int)
        for ind, row in train_data.iterrows():
            muts = get_mutations(row.serumHA, row.virusHA)
            
            if len(muts)==0:
                continue
            for mut in muts:
                mutation_counter[mut]+=1
        
        
        # make a list of mutations deemed relevant via frequency thresholds
        aa_frequency = frequency_aa(isolates)
        relevant_muts = []
        min_count     = 10
        min_freq      = 1.0*min_count/len(isolates)
        for mut, count in mutation_counter.items():
            pos = int(mut[1:-1])-1
            aa1, aa2 = mut[0],mut[-1]
            if count>min_count and \
                aa_frequency[(aa1, pos)]>min_freq and \
                aa_frequency[(aa2, pos)]>min_freq:
                    relevant_muts.append(mut)
        
        relevant_muts.sort(key = lambda x:int(x[1:-1]))
        
        return relevant_muts

def get_mutations(seq_serum, seq_virus):
    muts = []
    muts.extend([aa1+str(pos+1)+aa2 for pos, (aa1, aa2) in enumerate(zip(seq_serum, seq_virus)) if aa1!=aa2])
    
    return muts

def frequency_aa(isolates):
    sequences_expand = expand_sequences(isolates)
    
    aa_freq = defaultdict(int)
    
    # loop through the sites
    for pos in sequences_expand.columns:
        # aa count at each position
        aa_count = sequences_expand[pos].value_counts()
        
        # aa frequency
        for aa in aa_count.keys():
            aa_freq[(aa, pos)] = 1.0*aa_count[aa]/len(isolates)
    
    
    return aa_freq

def collapse_colinear_mutations(seq_graph, relevant_muts, colin_thres):
        n_genetic = len(relevant_muts)
        TT = seq_graph[:,:n_genetic].T
        mutation_clusters = [] 
        n_measurements = seq_graph.shape[0]
        
        # a greedy algorithm: if column is similar to existing cluster -> merge with cluster, else -> new cluster
        for col, mut in zip(TT, relevant_muts):
            col_found = False
            for cluster in mutation_clusters:
                # similarity is defined as number of measurements at which the cluster and column differ
                if np.sum(col==cluster[0])>=n_measurements-colin_thres:
                    cluster[1].append(mut)
                    col_found=True
                    print("adding",mut,"to cluster ",cluster[1]) 
                    break
            if not col_found:
                mutation_clusters.append([col, [mut]])
                    
        print("dimensions of old design matrix",seq_graph.shape)
        seq_graph = np.hstack((np.array([c[0] for c in mutation_clusters]).T, seq_graph[:,n_genetic:]))
        n_genetic = len(mutation_clusters)
        # use the first mutation of a cluster to index the effect
        # make a dictionary that maps this effect to the cluster
        mutation_clusters = {c[1][0]:c[1] for c in mutation_clusters}
        relevant_muts = [c[1][0] for c in mutation_clusters]
        print("dimensions of new design matrix",seq_graph.shape)
        
        return seq_graph, relevant_muts, mutation_clusters

def train_Nextflu_model(train_data, flu_type):
    SEED = 100
    random.seed(SEED)
    np.random.seed(SEED)
    train_data['serumName'] = train_data['serumName'].str.replace(' ', '')
    train_data['virusName'] = train_data['virusName'].str.replace(' ', '')
    train_data = train_data[train_data['Type'] == flu_type]

    group_cols = ['serumName', 'virusName','serumHA','virusHA']
    agg_dict = {col: 'first' for col in train_data.columns if col not in group_cols}
    agg_dict['label'] = 'mean'
    train_data = train_data.groupby(group_cols).agg(agg_dict).reset_index()

    viruses = train_data[['virusName', 'virusHA']].copy()
    viruses = viruses.drop_duplicates(['virusName'], keep='first', ignore_index=True)
    viruses.rename(columns={'virusName': 'isolateName', 'virusHA': 'sequence'}, inplace=True)
    viruses.sort_values(['isolateName'], inplace=True, ignore_index=True)

    sera = train_data[['serumName', 'serumHA']].copy()
    sera = sera.drop_duplicates(['serumName'], keep='first', ignore_index=True)
    sera.rename(columns={'serumName': 'isolateName', 'serumHA': 'sequence'}, inplace=True)
    sera.sort_values(['isolateName'], inplace=True, ignore_index=True)

    isolates = pd.concat((viruses, sera), ignore_index=True)
    isolates = isolates.drop_duplicates(['isolateName'], keep='first', ignore_index=True)
    isolates.sort_values(['isolateName'], inplace=True, ignore_index=True)

    sequences_expand = expand_sequences(isolates)
    aa_freq = defaultdict(int)

        # loop through the sites
    for pos in sequences_expand.columns:
        # aa count at each position
        aa_count = sequences_expand[pos].value_counts()
            
        # aa frequency
        for aa in aa_count.keys():
            aa_freq[(aa, pos)] = 1.0*aa_count[aa]/len(isolates)

    seq_serum = train_data['serumHA']
    seq_virus = train_data['virusHA']

    muts = []
    muts.extend([aa1+str(pos+1)+aa2 for pos, (aa1, aa2) in enumerate(zip(seq_serum, seq_virus)) if aa1!=aa2])

    seq_graph = []
    HI_dist   = []
    relevant_muts = relevant_mutations(train_data, isolates)
    # parameters of the model
    n_genetic = len(relevant_muts)
    n_sera = len(sera)
    n_v = len(viruses)
    n_params  = n_genetic + n_sera + n_v

    # loop over all measurements and encode the HI model as [0,1,0,1,0,0..] vector:
    # 1-> mutation present, 0 not present, same for serum and virus effects
    for ind, row in train_data.iterrows():
        if not np.isnan(row.label):
            muts = get_mutations(row.serumHA, row.virusHA)
            if len(muts)==0:
                continue
            tmp = np.zeros(n_params) # zero vector, ones will be filled in
            
            # determine branch indices on path
            mutation_indices = np.unique([relevant_muts.index(mut) for mut in muts if mut in relevant_muts])
            if len(mutation_indices): tmp[mutation_indices] = 1
            
            # add serum effect
            tmp[n_genetic+sera.index[sera.isolateName==row.serumName][0]] = 1
            
            # add virus effect
            tmp[n_genetic+n_sera+viruses.index[viruses.isolateName==row.virusName][0]] = 1
            
            # append model and HI_Dist value to lists seq_graph and HI_dist, respectively
            seq_graph.append(tmp)
            HI_dist.append(row.label)

    # convert to numpy arrays
    HI_dist   = np.array(HI_dist)
    seq_graph = np.array(seq_graph)

    # collapse colinear mutations
    colin_thres = None
    if colin_thres is not None:
        seq_graph, relevant_muts, mutation_clusters = collapse_colinear_mutations(seq_graph, relevant_muts, colin_thres)

    n_genetic = len(relevant_muts)
    n_params  = seq_graph.shape[1]

    # save product of tree graph with its transpose for future use
    TgT = np.dot(seq_graph.T, seq_graph)

    '''
    non-negative fit, branch terms L1 regularized, avidity terms L2 regularized
    '''

    lam_pot = 0.2
    lam_avi = 2
    lam_HI = 1
    # set up the quadratic matrix containing the deviation term (linear xterm below)
    # and the l2-regulatization of the avidities and potencies
    P1 = np.zeros((n_params,n_params))
    P1[:n_params, :n_params] = TgT
    for ii in range(n_genetic, n_genetic+n_sera):
        P1[ii,ii] += lam_pot
    for ii in range(n_genetic+n_sera, n_params):
        P1[ii,ii] += lam_avi
    P = matrix(P1)

    # set up cost for auxillary parameter and the linear cross-term
    q1 = np.zeros(n_params)
    q1[:n_params] = -np.dot(HI_dist, seq_graph)
    q1[:n_genetic] += lam_HI
    q = matrix(q1)

    # set up linear constraint matrix to enforce positivity of the
    # dHIs and bounding of dHI by the auxillary parameter
    h = matrix(np.zeros(n_genetic))   # Gw <=h
    G1 = np.zeros((n_genetic,n_params))
    G1[:n_genetic, :n_genetic] = -np.eye(n_genetic)
    G = matrix(G1)

    W = solvers.qp(P,q,G,h)

    params = np.array([x for x in W['x']])[:n_params]
    '''
    map substitution effects, serum potency and virus avidity
    '''
    mutation_effects={}
    for mi, mut in enumerate(relevant_muts):
        mutation_effects[mut] = params[mi]


    serum_potency = {serum:params[n_genetic+ii] for ii, serum in enumerate(sera.isolateName)}

    virus_avidity = {strain:params[n_genetic+n_sera+ii] for ii, strain in enumerate(viruses.isolateName)}

    model = {'mutation_effects': mutation_effects, 'serum_potency': serum_potency, 'virus_avidity': virus_avidity}

    return model

### titer test

In [34]:
Nextflu_train = pd.read_csv('../../../data/data_40/titer/train.csv',index_col=False)
Nextflu_test = pd.read_csv('../../../data/data_40/titer/test.csv')
H1N1_test = Nextflu_test[Nextflu_test['Type'] == 'H1N1']
H3N2_test = Nextflu_test[Nextflu_test['Type'] == 'H3N2']

In [ ]:
H1N1_model = train_Nextflu_model(Nextflu_train, 'H1N1')
H3N2_model = train_Nextflu_model(Nextflu_train, 'H3N2')

In [ ]:
H1N1_prediction = nextflu_predict(H1N1_test, H1N1_model['mutation_effects'], serum_potency=H1N1_model['serum_potency'], virus_avidity=H1N1_model['virus_avidity'])
H3N2_prediction = nextflu_predict(H3N2_test, H3N2_model['mutation_effects'], serum_potency=H3N2_model['serum_potency'], virus_avidity=H3N2_model['virus_avidity'])
Nextflu_test.loc[Nextflu_test['Type'] == 'H1N1', 'pred_with_name'] = H1N1_prediction
Nextflu_test.loc[Nextflu_test['Type'] == 'H3N2', 'pred_with_name'] = H3N2_prediction
result_with_name = print_exams(Nextflu_test['pred_with_name'], Nextflu_test['label'])

MAE: 0.59709
MSE: 0.62343
pearson correlation: 0.91687
spearman correlation: 0.90071
R2_score: 0.84063


In [37]:
temp_test = Nextflu_test.copy()
temp_test.loc[:, 'serumName'] = ''
temp_test.loc[:, 'virusName'] = ''
H1N1_prediction = nextflu_predict(temp_test[temp_test['Type'] == 'H1N1'], H1N1_model['mutation_effects'], serum_potency=H1N1_model['serum_potency'], virus_avidity=H1N1_model['virus_avidity'])
H3N2_prediction = nextflu_predict(temp_test[temp_test['Type'] == 'H3N2'], H3N2_model['mutation_effects'], serum_potency=H3N2_model['serum_potency'], virus_avidity=H3N2_model['virus_avidity'])
Nextflu_test.loc[Nextflu_test['Type'] == 'H1N1', 'pred_without_name'] = H1N1_prediction
Nextflu_test.loc[Nextflu_test['Type'] == 'H3N2', 'pred_without_name'] = H3N2_prediction
All_result = print_exams(Nextflu_test['pred_without_name'], Nextflu_test['label'])

MAE: 0.94166
MSE: 1.57047
pearson correlation: 0.78245
spearman correlation: 0.74422
R2_score: 0.59854


In [ ]:
with open("../../../trained_model/Nextflu/H1N1_titer.json", "w", encoding="utf-8") as f:
    json.dump(H1N1_model, f, ensure_ascii=False, indent=2)
with open("../../../trained_model/Nextflu/H3N2_titer.json", "w", encoding="utf-8") as f:
    json.dump(H3N2_model, f, ensure_ascii=False, indent=2)

In [10]:
Nextflu_test.to_csv('../../../Figure/Fig2/Nextflu_titer.csv', index=False)

### strain test

In [14]:
Nextflu_train = pd.read_csv('../../../data/data_40/strain/train.csv',index_col=False)
Nextflu_test = pd.read_csv('../../../data/data_40/strain/test.csv', index_col=False)
H1N1_test = Nextflu_test[Nextflu_test['Type'] == 'H1N1']
H3N2_test = Nextflu_test[Nextflu_test['Type'] == 'H3N2']

In [15]:
H1N1_model = train_Nextflu_model(Nextflu_train, 'H1N1')
H3N2_model = train_Nextflu_model(Nextflu_train, 'H3N2')

     pcost       dcost       gap    pres   dres
 0: -5.3544e+04 -5.3451e+04  4e+03  5e+01  2e-03
 1: -5.3455e+04 -5.3512e+04  1e+03  1e+01  4e-04
 2: -5.3400e+04 -5.3452e+04  6e+02  7e+00  2e-04
 3: -5.3313e+04 -5.3409e+04  4e+02  3e+00  1e-04
 4: -5.3297e+04 -5.3362e+04  2e+02  1e+00  4e-05
 5: -5.3296e+04 -5.3329e+04  7e+01  3e-01  1e-05
 6: -5.3304e+04 -5.3311e+04  7e+00  1e-15  6e-16
 7: -5.3309e+04 -5.3309e+04  5e-01  1e-15  8e-16
 8: -5.3309e+04 -5.3309e+04  2e-02  1e-15  6e-16
Optimal solution found.
     pcost       dcost       gap    pres   dres
 0: -1.3203e+05 -1.3207e+05  7e+03  6e+01  1e-03
 1: -1.3197e+05 -1.3206e+05  2e+03  2e+01  4e-04
 2: -1.3186e+05 -1.3195e+05  1e+03  7e+00  1e-04
 3: -1.3179e+05 -1.3186e+05  5e+02  3e+00  5e-05
 4: -1.3176e+05 -1.3180e+05  2e+02  1e+00  2e-05
 5: -1.3174e+05 -1.3176e+05  8e+01  3e-01  5e-06
 6: -1.3174e+05 -1.3175e+05  1e+01  1e-02  3e-07
 7: -1.3174e+05 -1.3174e+05  3e+00  2e-03  3e-08
 8: -1.3174e+05 -1.3174e+05  3e+00  1e-03  3e-0

In [16]:
H1N1_prediction = nextflu_predict(H1N1_test, H1N1_model['mutation_effects'], serum_potency=H1N1_model['serum_potency'], virus_avidity=H1N1_model['virus_avidity'])
H3N2_prediction = nextflu_predict(H3N2_test, H3N2_model['mutation_effects'], serum_potency=H3N2_model['serum_potency'], virus_avidity=H3N2_model['virus_avidity'])
Nextflu_test.loc[Nextflu_test['Type'] == 'H1N1', 'pred_with_name'] = H1N1_prediction
Nextflu_test.loc[Nextflu_test['Type'] == 'H3N2', 'pred_with_name'] = H3N2_prediction
result = print_exams(Nextflu_test['pred_with_name'], Nextflu_test['label'])

MAE: 0.76320
MSE: 1.06242
pearson correlation: 0.85154
spearman correlation: 0.82362
R2_score: 0.72493


In [17]:
temp_test = Nextflu_test.copy()
temp_test.loc[:, 'serumName'] = ''
temp_test.loc[:, 'virusName'] = ''
H1N1_prediction = nextflu_predict(temp_test[temp_test['Type'] == 'H1N1'], H1N1_model['mutation_effects'], serum_potency=H1N1_model['serum_potency'], virus_avidity=H1N1_model['virus_avidity'])
H3N2_prediction = nextflu_predict(temp_test[temp_test['Type'] == 'H3N2'], H3N2_model['mutation_effects'], serum_potency=H3N2_model['serum_potency'], virus_avidity=H3N2_model['virus_avidity'])
Nextflu_test.loc[temp_test['Type'] == 'H1N1', 'pred_without_name'] = H1N1_prediction
Nextflu_test.loc[temp_test['Type'] == 'H3N2', 'pred_without_name'] = H3N2_prediction
result = print_exams(Nextflu_test['pred_without_name'], Nextflu_test['label'])

MAE: 1.00833
MSE: 1.85546
pearson correlation: 0.73436
spearman correlation: 0.69973
R2_score: 0.51960


In [18]:
Nextflu_test.to_csv('../../../Figure/Fig2/Nextflu_strain.csv', index=False)

### serum test

In [19]:
Nextflu_train = pd.read_csv('../../../data/data_40/serum/train.csv',index_col=False)
Nextflu_test = pd.read_csv('../../../data/data_40/serum/test.csv', index_col=False)
H1N1_test = Nextflu_test[Nextflu_test['Type'] == 'H1N1']
H3N2_test = Nextflu_test[Nextflu_test['Type'] == 'H3N2']

In [20]:
H1N1_model = train_Nextflu_model(Nextflu_train, 'H1N1')
H3N2_model = train_Nextflu_model(Nextflu_train, 'H3N2')

     pcost       dcost       gap    pres   dres
 0: -5.8399e+04 -5.8299e+04  4e+03  5e+01  2e-03
 1: -5.8322e+04 -5.8363e+04  1e+03  2e+01  6e-04
 2: -5.8236e+04 -5.8299e+04  6e+02  7e+00  2e-04
 3: -5.8146e+04 -5.8255e+04  5e+02  4e+00  1e-04
 4: -5.8127e+04 -5.8194e+04  2e+02  1e+00  4e-05
 5: -5.8103e+04 -5.8164e+04  1e+02  4e-01  1e-05
 6: -5.8124e+04 -5.8137e+04  1e+01  2e-02  5e-07
 7: -5.8131e+04 -5.8133e+04  2e+00  9e-04  3e-08
 8: -5.8132e+04 -5.8132e+04  6e-02  1e-05  5e-10
 9: -5.8132e+04 -5.8132e+04  2e-03  2e-07  7e-12
10: -5.8132e+04 -5.8132e+04  7e-05  2e-09  7e-14
Optimal solution found.
     pcost       dcost       gap    pres   dres
 0: -1.3331e+05 -1.3334e+05  7e+03  7e+01  1e-03
 1: -1.3325e+05 -1.3334e+05  2e+03  2e+01  4e-04
 2: -1.3312e+05 -1.3323e+05  1e+03  8e+00  2e-04
 3: -1.3304e+05 -1.3314e+05  7e+02  3e+00  6e-05
 4: -1.3300e+05 -1.3306e+05  3e+02  1e+00  2e-05
 5: -1.3298e+05 -1.3302e+05  1e+02  4e-01  8e-06
 6: -1.3298e+05 -1.3300e+05  3e+01  7e-02  1e-0

In [24]:
H1N1_prediction = nextflu_predict(H1N1_test, H1N1_model['mutation_effects'], serum_potency=H1N1_model['serum_potency'], virus_avidity=H1N1_model['virus_avidity'])
H3N2_prediction = nextflu_predict(H3N2_test, H3N2_model['mutation_effects'], serum_potency=H3N2_model['serum_potency'], virus_avidity=H3N2_model['virus_avidity'])
Nextflu_test.loc[Nextflu_test['Type'] == 'H1N1', 'pred_with_name'] = H1N1_prediction
Nextflu_test.loc[Nextflu_test['Type'] == 'H3N2', 'pred_with_name'] = H3N2_prediction
result = print_exams(Nextflu_test['pred_with_name'], Nextflu_test['label'])

MAE: 0.86197
MSE: 1.22519
pearson correlation: 0.85020
spearman correlation: 0.86211
R2_score: 0.68799


In [25]:
temp_test = Nextflu_test.copy()
temp_test.loc[:, 'serumName'] = ''
temp_test.loc[:, 'virusName'] = ''
H1N1_prediction = nextflu_predict(temp_test[temp_test['Type'] == 'H1N1'], H1N1_model['mutation_effects'], serum_potency=H1N1_model['serum_potency'], virus_avidity=H1N1_model['virus_avidity'])
H3N2_prediction = nextflu_predict(temp_test[temp_test['Type'] == 'H3N2'], H3N2_model['mutation_effects'], serum_potency=H3N2_model['serum_potency'], virus_avidity=H3N2_model['virus_avidity'])
Nextflu_test.loc[temp_test['Type'] == 'H1N1', 'pred_without_name'] = H1N1_prediction
Nextflu_test.loc[temp_test['Type'] == 'H3N2', 'pred_without_name'] = H3N2_prediction
result = print_exams(Nextflu_test['pred_without_name'], Nextflu_test['label'])

MAE: 1.07467
MSE: 1.92276
pearson correlation: 0.72651
spearman correlation: 0.75567
R2_score: 0.51035


In [26]:
Nextflu_test.to_csv('../../../Figure/Fig2/Nextflu_serum.csv', index=False)

### future 41 test

In [10]:
Crick_41 = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/Crick_41_mapped.csv')
group_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']
agg_dict = {c: 'first' for c in Crick_41.columns if c not in group_columns}
agg_dict['label'] = 'mean'
Nextflu_test = Crick_41.groupby(group_columns).agg(agg_dict).reset_index()

In [11]:
H1N1_test = Nextflu_test[Nextflu_test['Type'] == 'H1N1']
H3N2_test = Nextflu_test[Nextflu_test['Type'] == 'H3N2']

In [12]:
H1N1_prediction = nextflu_predict(H1N1_test, H1N1_model['mutation_effects'], serum_potency=H1N1_model['serum_potency'], virus_avidity=H1N1_model['virus_avidity'])
H3N2_prediction = nextflu_predict(H3N2_test, H3N2_model['mutation_effects'], serum_potency=H3N2_model['serum_potency'], virus_avidity=H3N2_model['virus_avidity'])
Nextflu_test.loc[Nextflu_test['Type'] == 'H1N1', 'pred_with_name'] = H1N1_prediction
Nextflu_test.loc[Nextflu_test['Type'] == 'H3N2', 'pred_with_name'] = H3N2_prediction
result = print_exams(Nextflu_test['pred_with_name'], Nextflu_test['label'])

MAE: 0.84923
MSE: 1.06800
pearson correlation: 0.62355
spearman correlation: 0.59994
R2_score: 0.24198


In [13]:
temp_test = Nextflu_test.copy()
temp_test.loc[:, 'serumName'] = ''
temp_test.loc[:, 'virusName'] = ''
H1N1_prediction = nextflu_predict(temp_test[temp_test['Type'] == 'H1N1'], H1N1_model['mutation_effects'], serum_potency=H1N1_model['serum_potency'], virus_avidity=H1N1_model['virus_avidity'])
H3N2_prediction = nextflu_predict(temp_test[temp_test['Type'] == 'H3N2'], H3N2_model['mutation_effects'], serum_potency=H3N2_model['serum_potency'], virus_avidity=H3N2_model['virus_avidity'])
Nextflu_test.loc[temp_test['Type'] == 'H1N1', 'pred_without_name'] = H1N1_prediction
Nextflu_test.loc[temp_test['Type'] == 'H3N2', 'pred_without_name'] = H3N2_prediction
result = print_exams(Nextflu_test['pred_without_name'], Nextflu_test['label'])

MAE: 0.77944
MSE: 0.92891
pearson correlation: 0.61549
spearman correlation: 0.58749
R2_score: 0.34069


In [15]:
Nextflu_test.to_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/Figure/Fig2/Nextflu_41.csv')

### CDC test

In [17]:
CDC = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/CDC_all.csv')
CDC = CDC.loc[CDC['virusDate'] <= '2023-08-31',:]

group_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']
agg_dict = {c: 'first' for c in CDC.columns if c not in group_columns}
agg_dict['label'] = 'mean'
Nextflu_test = CDC.groupby(group_columns).agg(agg_dict).reset_index()

In [18]:
H1N1_test = Nextflu_test[Nextflu_test['Type'] == 'H1N1']
H3N2_test = Nextflu_test[Nextflu_test['Type'] == 'H3N2']

In [19]:
H1N1_prediction = nextflu_predict(H1N1_test, H1N1_model['mutation_effects'], serum_potency=H1N1_model['serum_potency'], virus_avidity=H1N1_model['virus_avidity'])
H3N2_prediction = nextflu_predict(H3N2_test, H3N2_model['mutation_effects'], serum_potency=H3N2_model['serum_potency'], virus_avidity=H3N2_model['virus_avidity'])
Nextflu_test.loc[Nextflu_test['Type'] == 'H1N1', 'pred_with_name'] = H1N1_prediction
Nextflu_test.loc[Nextflu_test['Type'] == 'H3N2', 'pred_with_name'] = H3N2_prediction
result = print_exams(Nextflu_test['pred_with_name'], Nextflu_test['label'])

MAE: 1.08735
MSE: 2.10493
pearson correlation: 0.72321
spearman correlation: 0.65178
R2_score: 0.50555


In [20]:
temp_test = Nextflu_test.copy()
temp_test.loc[:, 'serumName'] = ''
temp_test.loc[:, 'virusName'] = ''
H1N1_prediction = nextflu_predict(temp_test[temp_test['Type'] == 'H1N1'], H1N1_model['mutation_effects'], serum_potency=H1N1_model['serum_potency'], virus_avidity=H1N1_model['virus_avidity'])
H3N2_prediction = nextflu_predict(temp_test[temp_test['Type'] == 'H3N2'], H3N2_model['mutation_effects'], serum_potency=H3N2_model['serum_potency'], virus_avidity=H3N2_model['virus_avidity'])
Nextflu_test.loc[temp_test['Type'] == 'H1N1', 'pred_without_name'] = H1N1_prediction
Nextflu_test.loc[temp_test['Type'] == 'H3N2', 'pred_without_name'] = H3N2_prediction
result = print_exams(Nextflu_test['pred_without_name'], Nextflu_test['label'])

MAE: 1.00642
MSE: 1.87916
pearson correlation: 0.74780
spearman correlation: 0.66002
R2_score: 0.55858


In [21]:
Nextflu_test.to_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/Figure/Fig2/Nextflu_CDC.csv')

### CNIC test

In [23]:
CNIC = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/CNIC_all.csv')
CNIC = CNIC.loc[CNIC['virusDate'] <= '2023-08-31',:]

group_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']
agg_dict = {c: 'first' for c in CNIC.columns if c not in group_columns}
agg_dict['label'] = 'mean'
Nextflu_test = CNIC.groupby(group_columns).agg(agg_dict).reset_index()

In [24]:
H1N1_test = Nextflu_test[Nextflu_test['Type'] == 'H1N1']
H3N2_test = Nextflu_test[Nextflu_test['Type'] == 'H3N2']

In [25]:
H1N1_prediction = nextflu_predict(H1N1_test, H1N1_model['mutation_effects'], serum_potency=H1N1_model['serum_potency'], virus_avidity=H1N1_model['virus_avidity'])
H3N2_prediction = nextflu_predict(H3N2_test, H3N2_model['mutation_effects'], serum_potency=H3N2_model['serum_potency'], virus_avidity=H3N2_model['virus_avidity'])
Nextflu_test.loc[Nextflu_test['Type'] == 'H1N1', 'pred_with_name'] = H1N1_prediction
Nextflu_test.loc[Nextflu_test['Type'] == 'H3N2', 'pred_with_name'] = H3N2_prediction
result = print_exams(Nextflu_test['pred_with_name'], Nextflu_test['label'])

MAE: 1.29024
MSE: 2.91401
pearson correlation: 0.67633
spearman correlation: 0.68697
R2_score: 0.41401


In [26]:
temp_test = Nextflu_test.copy()
temp_test.loc[:, 'serumName'] = ''
temp_test.loc[:, 'virusName'] = ''
H1N1_prediction = nextflu_predict(temp_test[temp_test['Type'] == 'H1N1'], H1N1_model['mutation_effects'], serum_potency=H1N1_model['serum_potency'], virus_avidity=H1N1_model['virus_avidity'])
H3N2_prediction = nextflu_predict(temp_test[temp_test['Type'] == 'H3N2'], H3N2_model['mutation_effects'], serum_potency=H3N2_model['serum_potency'], virus_avidity=H3N2_model['virus_avidity'])
Nextflu_test.loc[temp_test['Type'] == 'H1N1', 'pred_without_name'] = H1N1_prediction
Nextflu_test.loc[temp_test['Type'] == 'H3N2', 'pred_without_name'] = H3N2_prediction
result = print_exams(Nextflu_test['pred_without_name'], Nextflu_test['label'])

MAE: 1.40020
MSE: 3.39113
pearson correlation: 0.64819
spearman correlation: 0.66479
R2_score: 0.31806


In [27]:
Nextflu_test.to_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/Figure/Fig2/Nextflu_CNIC.csv')